# 01 · Forward model — worked solutions

These solutions demonstrate one controlled way to answer each exercise. The
qualitative interpretation matters as much as the printed maximum values. Try
the experiments in the main notebook before reading onward.

In [1]:
import numpy as np

import geodef

## Shared setup

We use the same fault dimensions, slip pattern, and regular surface grid as the
chapter. Keeping those choices fixed lets each exercise isolate one change.

In [2]:
base_fault = geodef.Fault.planar(
    lat=-2.0, lon=100.0, depth=30e3, strike=315.0, dip=15.0,
    length=180e3, width=60e3, n_length=12, n_width=6,
)

In [3]:
along = np.arange(base_fault.n_patches) % 12
down = np.arange(base_fault.n_patches) // 12
slip = 3 * np.exp(-((along - 5.5) / 3) ** 2 - ((down - 2.5) / 1.5) ** 2)

In [4]:
axis_km = np.linspace(-220, 220, 61)
east_km, north_km = np.meshgrid(axis_km, axis_km)
surface = base_fault.frame.to_geographic(
    east=east_km * 1000, north=north_km * 1000, up=0
)
lat, lon = surface[..., 1].ravel(), surface[..., 0].ravel()

## 1. Depth experiment

We predict a broader, weaker surface field for the deeper source because the
elastic response spreads farther before reaching the surface. The peak vector
norm confirms the amplitude prediction.

In [5]:
depth_peaks = {}
for depth_m in (20e3, 40e3):
    test_fault = geodef.Fault.planar(
        lat=-2.0, lon=100.0, depth=depth_m, strike=315.0, dip=15.0,
        length=180e3, width=60e3, n_length=12, n_width=6,
    )
    components = test_fault.displacement(lat, lon, 0.0, slip)
    depth_peaks[depth_m / 1000] = float(
        np.max(np.linalg.norm(components, axis=0))
    )
print(depth_peaks)

{20.0: 0.6817082851565566, 40.0: 0.38210335154186215}


A single maximum cannot demonstrate greater width by itself. Inspecting the
profiles or measuring the distance over which displacement remains appreciable
is necessary for that part of the interpretation.

## 2. Dip experiment

Dip changes the orientation of the slip vector and the buried rectangle. We
therefore compare all three component maxima rather than collapsing the result
immediately to one vector norm.

In [6]:
dip_peaks = {}
for dip in (15.0, 35.0, 60.0):
    test_fault = geodef.Fault.planar(
        lat=-2.0, lon=100.0, depth=30e3, strike=315.0, dip=dip,
        length=180e3, width=60e3, n_length=12, n_width=6,
    )
    components = test_fault.displacement(lat, lon, 0.0, slip)
    dip_peaks[dip] = [
        float(np.max(np.abs(values))) for values in components
    ]
print(dip_peaks)

{15.0: [0.20893683541139962, 0.20894437590595277, 0.46144300460865795], 35.0: [0.15354354506457843, 0.15355258883304593, 0.6163134410116123], 60.0: [0.2486612070446116, 0.24866527905526994, 0.7650404513928526]}


The length and width are fixed, but their projection and depth range change with
dip. Component maxima need not vary monotonically because orientation and source
distance change together.

## 3. Moment scaling

Doubling slip should double moment and every displacement component. Magnitude
should increase by $(2/3)\log_{10}(2)$, about 0.20.

In [7]:
base_components = base_fault.displacement(lat, lon, 0.0, slip)
double_components = base_fault.displacement(lat, lon, 0.0, 2 * slip)
moment_ratio = base_fault.moment(2 * slip) / base_fault.moment(slip)
magnitude_change = base_fault.magnitude(2 * slip) - base_fault.magnitude(slip)

In [8]:
displacement_ratio = np.max(np.abs(double_components[2])) / np.max(
    np.abs(base_components[2])
)
print(moment_ratio, displacement_ratio, magnitude_change)

2.0 2.0 0.2006866637759881


## 4. Different mechanism

Using the bell-shaped distribution as strike slip changes the symmetry because
the discontinuity is now parallel to strike instead of up the dipping plane.
The vertical component is generally smaller and changes sign in a different
pattern, while horizontal motion develops the familiar strike-slip lobes.

In [9]:
strike_components = base_fault.displacement(lat, lon, slip, 0.0)
strike_maxima = [float(np.max(np.abs(values))) for values in strike_components]
dip_maxima = [float(np.max(np.abs(values))) for values in base_components]
print("strike-slip maxima:", strike_maxima)
print("dip-slip maxima:", dip_maxima)

strike-slip maxima: [0.26395222197987345, 0.26393906381019616, 0.22118962972669726]
dip-slip maxima: [0.20893683541139962, 0.20894437590595277, 0.46144300460865795]


Maxima alone do not show symmetry. The correct completion of the exercise also
reproduces the three component maps and describes where each sign occurs.

## 5. Superposition

We divide the original model into 40% and 60% pieces. Their patchwise sum is the
original slip, so linearity predicts exact agreement up to floating-point
roundoff.

In [10]:
first = base_fault.displacement(lat, lon, 0.0, 0.4 * slip)
second = base_fault.displacement(lat, lon, 0.0, 0.6 * slip)
combined = tuple(a + b for a, b in zip(first, second))
print([np.allclose(total, expected)
       for total, expected in zip(combined, base_components)])

[True, True, True]


## Closing interpretation

Controlled experiments separate causes. Depth primarily changes attenuation and
spatial width; dip and mechanism rotate the source and redistribute component
amplitudes; slip amplitude scales displacement and moment linearly. These
expectations become diagnostic tools when later inverse results behave oddly.